# 01 · Ingesta y arquitectura medallion (Bronze → Silver → Gold)

Descarga las cuatro fuentes del proyecto y construye las tres capas del modelo medallion con PySpark. En cada capa se muestra **qué hace Spark con los datos**: el plan físico que elige (`explain`) y lo que registra la Spark UI (jobs, tareas, MB leídos y escritos, shuffle).

| Fuente | Institución | Formato | Periodicidad |
| :--- | :--- | :--- | :--- |
| Ingresos y gastos del Gobierno Federal | SHCP | CSV | Mensual |
| Ingresos tributarios | SAT | XLS → CSV | Mensual |
| IGAE (base 2018) | INEGI | CSV (zip) | Mensual |
| INPC | INEGI | CSV (zip) | Mensual |

| Capa | Carpeta | Contenido | Código |
| :--- | :--- | :--- | :--- |
| Raw | `data/raw/` | Archivos tal como se descargan | Este notebook (secciones 1–5) |
| **Bronze** | `data/bronze/` | Mismas filas y columnas en Parquet, con archivo de origen y fecha de ingesta | `src/ingestion/loader.py` |
| **Silver** | `data/silver/` | Una tabla limpia y validada por fuente, con llave común `fecha` | `src/transformations/clean.py`, `src/quality/validators.py` |
| **Gold** | `data/gold/` | Dataset maestro analítico: 1 fila = 1 mes, 2015-01 a 2025-03 | `src/transformations/integrate.py` |

Las funciones `capa_bronze`, `capa_silver` y `capa_gold` son las mismas que ejecuta `python -m src.pipeline`; aquí se llaman una por una para revisar a Spark entre capas.

**Spark UI:** mientras corre el notebook, la URL aparece en la sección 6 (normalmente http://localhost:4040); los jobs llevan la etiqueta de su capa. Las ejecuciones terminadas se pueden reabrir en el History Server con el *event log* de `spark-events/` (instrucciones en el README).

> Ejecutar desde `notebooks/`. Spark 3.5 requiere **Java 17**. Los datos no se versionan (`data/` está en `.gitignore`); detalle de cada fuente en [docs/fuentes.md](../docs/fuentes.md).

In [1]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

import shutil
import ssl
import sys
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.ingestion.loader import (
    RAW_DIR as RAW, BRONZE_DIR, SILVER_DIR, GOLD_DIR,
    get_spark, extract_sources, resolve_raw_paths, to_bronze, read_layer,
)
from src.transformations.clean import clean_all
from src.transformations.integrate import build_master_dataset
from src.pipeline import capa_bronze, capa_silver, capa_gold
from src.monitoring import entorno, metricas

RAW.mkdir(parents=True, exist_ok=True)
tamano_mb = lambda ruta: sum(f.stat().st_size for f in Path(ruta).rglob("*") if f.is_file()) / 1e6 if Path(ruta).is_dir() else Path(ruta).stat().st_size / 1e6
print("Datos crudos en:", RAW)

Datos crudos en: /home/mauriciodiaz/MCD/1/PGBD/INtegrador/Integrador-PGBD-26A/data/raw


In [2]:
# ============================================================
# 2. FUENTES
# ============================================================

FUENTES = {
    "ingreso_gasto_finan.csv": "https://www.secciones.hacienda.gob.mx/work/models/estadisticas_oportunas/datos_abiertos_eopf/ingreso_gasto_finan.csv",
    "IngresosTributarios.xls": "https://wu1agsprosta001.blob.core.windows.net/agsc-publicaciones/Datos_abiertos/Cifras_SAT/Documents/IngresosTributarios.xls",
    "igae_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/igae/2018/datosabiertos/igae_mensual_csv.zip",
    "inpc_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/inpc/2018a/datosabiertos/conjunto_de_datos_inpc_indicador_mensual_csv.zip",
}

In [3]:
# ============================================================
# 3. DESCARGA
# ============================================================
#
# El servidor de la SHCP no envía el certificado intermedio
# correcto (Let's Encrypt YR1). En lugar de desactivar la
# verificación SSL, descargamos ese intermedio y lo agregamos
# al contexto: la conexión sigue verificada contra la raíz.
#
# Si el archivo ya existe no se vuelve a descargar.
# Para actualizar una fuente, borrar su archivo de data/raw/.
# ============================================================

ctx = ssl.create_default_context()
yr1 = urllib.request.urlopen("http://yr1.i.lencr.org/", timeout=60).read()
ctx.load_verify_locations(cadata=ssl.DER_cert_to_PEM_cert(yr1))

for nombre, url in FUENTES.items():
    destino = RAW / nombre
    if not destino.exists():
        tmp = destino.with_suffix(".part")
        with urllib.request.urlopen(url, context=ctx, timeout=600) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
        tmp.rename(destino)  # evita dejar archivos a medias si falla la descarga
    print(f"{nombre:28} {destino.stat().st_size / 1e6:8.1f} MB")

ingreso_gasto_finan.csv         224.9 MB
IngresosTributarios.xls           0.2 MB
igae_mensual_csv.zip              0.4 MB
inpc_mensual_csv.zip              0.0 MB


In [4]:
# ============================================================
# 4. DESCOMPRESIÓN (INEGI)
# ============================================================

for zip_name in ["igae_mensual_csv.zip", "inpc_mensual_csv.zip"]:
    with zipfile.ZipFile(RAW / zip_name) as z:
        z.extractall(RAW / zip_name.removesuffix(".zip"))

RUTA_IGAE = next((RAW / "igae_mensual_csv" / "conjunto_de_datos").glob("conjunto_de_datos_igae_*.csv"))
RUTA_INPC = RAW / "inpc_mensual_csv" / "conjunto_de_datos" / "conjunto_de_datos_inpc_mensual.csv"
print(RUTA_IGAE.name)
print(RUTA_INPC.name)

conjunto_de_datos_igae_igae2025_03.csv
conjunto_de_datos_inpc_mensual.csv


In [5]:
# ============================================================
# 5. SAT: XLS → CSV
# ============================================================
#
# Spark no lee Excel de forma nativa. El archivo tiene:
# - título en las filas 1-4;
# - encabezado en la fila 5 (y subencabezados de IEPS en la 6);
# - notas al pie al final.
#
# Solo cambiamos de formato y quitamos notas; la limpieza
# (meses en texto, cifras preliminares) va en transformaciones.
# ============================================================

sat_xls = pd.read_excel(RAW / "IngresosTributarios.xls", header=None).dropna(axis=1, how="all")

encabezado = sat_xls.iloc[5].fillna(sat_xls.iloc[4]).str.strip()
sat = sat_xls.iloc[6:].set_axis(encabezado, axis=1)
sat = sat[sat["Mes"].notna()]  # las notas al pie no tienen mes

RUTA_SAT = RAW / "ingresos_tributarios_sat.csv"
sat.to_csv(RUTA_SAT, index=False)

print(sat.shape)
sat.head()

(201, 22)


5,Año,Mes,Total de Ingresos tributarios,Impuesto Sobre la Renta,Impuesto al Valor Agregado,Impuesto Especial sobre Producción y Servicios,Gasolinas y diésel,Tabacos labrados,Alcohol,Cerveza y bebidas refrescantes,...,Bebidas energetizantes,Bebidas saborizadas,Alimentos no básicos con alta densidad calórica,Plaguicidas,Carbono,Importaciones,Exportaciones,Impuesto sobre Automóviles Nuevos,Impuesto por la Actividad de Exploración y Extracción de Hidrocarburos,Otros ingresos tributarios
6,2010,Enero,143806.723,71997.945,58816.947,2628.27,-6928.45,6808.447,734.741,1865.145,...,0,n.d.,n.d.,n.d.,n.d.,1988.947,0.004,559.868,n.d.,7814.742
7,2010,Febrero,99163.694,46098.298,36601.913,-1320.321,-4171.001,655.931,288.865,1343.903,...,0,n.d.,n.d.,n.d.,n.d.,1814.108,0.017,333.424,n.d.,15636.255
8,2010,Marzo,110803.305,55628.337,35756.921,-2675.345,-5954.753,339.342,843.49,1455.825,...,0,n.d.,n.d.,n.d.,n.d.,2111.401,0.064,349.452,n.d.,19632.475
9,2010,Abril,105036.462,53859.74,40199.949,-981.862,-5423.797,1621.642,301.466,1684.927,...,0,n.d.,n.d.,n.d.,n.d.,1767.535,0.008,389.153,n.d.,9801.939
10,2010,Mayo,90663.865,45604.199,37907.734,-1953.271,-6539.372,1656.855,352.475,1810.576,...,0,n.d.,n.d.,n.d.,n.d.,1991.99,0.031,335.969,n.d.,6777.213


## 6. Sesión de Spark: una instancia local con 12 núcleos

`get_spark()` (en `src/ingestion/loader.py`) crea la sesión con `master("local[*]")`: **un solo proceso JVM** hace de driver y executor, con un *slot* de tarea por núcleo lógico. La tabla muestra los recursos reales y el tamaño de lo que hay que procesar.

In [6]:
spark = get_spark("Integrador_01_Ingesta_Medallion")
spark.sparkContext.setLogLevel("ERROR")

display(pd.DataFrame(
    [(n, r.name, round(tamano_mb(r), 2)) for n, r in resolve_raw_paths().items()],
    columns=["fuente", "archivo crudo", "MB"],
))
pd.Series(entorno(spark), name="valor").to_frame()

,fuente,archivo crudo,MB
0,shcp,ingreso_gasto_finan.csv,224.93
1,sat,ingresos_tributarios_sat.csv,0.04
2,igae,conjunto_de_datos_igae_igae2025_03.csv,0.92
3,inpc,conjunto_de_datos_inpc_mensual.csv,0.40


,valor
master,local[*]
executors (procesos JVM),1
slots de tarea (núcleos),12
memoria unificada Spark (MB),456
spark.driver.memory,1g (por defecto)
spark.sql.shuffle.partitions,4
spark.sql.adaptive.enabled,true
spark.sql.files.maxPartitionBytes,134217728b
Spark UI,http://fedora:4040


> **Por qué una sola instancia local.** Todo el insumo pesa **~226 MB** y el **99.4 %** es el CSV de SHCP; cabe holgadamente en la memoria de una máquina con 16 GB.
> - `local[*]` crea **1 executor (el propio driver) con 12 slots de tarea**, uno por núcleo lógico. Un cluster agregaría arranque de executors, serialización y tráfico de red en cada shuffle, sin nada que ganar con este volumen: el pipeline completo tarda segundos.
> - **12 slots = 12 particiones de lectura del CSV grande** (sección 7): cada núcleo lee un trozo y la lectura termina en una sola ola de tareas. Con menos núcleos (`local[4]`), parte de la lectura esperaría turno.
> - **Memoria:** el driver usa su valor por defecto (1 GB; ~456 MB para ejecución y caché de Spark). En las mediciones el GC ocupó 1.9 % del tiempo de tareas y no hubo derrame a disco (ver [docs/guia_optimizacion.md](../docs/guia_optimizacion.md)).
> - **Cuándo escalar:** si el insumo superara la RAM (decenas de GB) o el tiempo dejara de ser de segundos, convendría un cluster con varios executors.

## 7. Capa Bronze: CSV crudo → Parquet sin transformar

Bronze guarda las fuentes **tal como llegan** (mismas filas y columnas) pero en Parquet, y agrega dos columnas de trazabilidad: `_archivo_origen` y `_fecha_ingesta`. Es la única capa que lee los CSV; todo lo demás parte de aquí.

**Cómo divide Spark el CSV en particiones.** La regla de `FilePartition.maxSplitBytes` es:

$$\text{maxSplitBytes} = \min\Big(\text{maxPartitionBytes},\ \max\big(\text{openCostInBytes},\ \tfrac{\text{bytes} + n_{\text{archivos}}\cdot\text{openCost}}{\text{defaultParallelism}}\big)\Big)$$

Con los valores por defecto (128 MB y 4 MB) y `defaultParallelism` = 12 núcleos, la celda siguiente calcula cuántas particiones (tareas de lectura) debe producir cada archivo y lo compara con lo que Spark realmente hace.

In [7]:
raw = extract_sources(spark)
nucleos = spark.sparkContext.defaultParallelism
filas = []
for nombre, ruta in resolve_raw_paths().items():
    total = ruta.stat().st_size
    split = min(128 * 1024**2, max(4 * 1024**2, (total + 4 * 1024**2) / nucleos))
    filas.append({"fuente": nombre, "MB": round(total / 1e6, 2), "split calculado (MB)": round(split / 1e6, 1),
                  "particiones esperadas": -(-total // int(split)),
                  "particiones reales": raw[nombre].rdd.getNumPartitions()})
pd.DataFrame(filas)

,fuente,MB,split calculado (MB),particiones esperadas,particiones reales
0,shcp,224.93,19.1,12,12
1,sat,0.04,4.2,1,1
2,igae,0.92,4.2,1,1
3,inpc,0.40,4.2,1,1


In [8]:
# Plan físico de la lectura de SHCP hacia Bronze
to_bronze(raw)["shcp"].explain("formatted")

== Physical Plan ==
* Project (3)
+- * Project (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [16]: [CICLO#0, MES#1, CLAVE_DE_CONCEPTO#2, NOMBRE#3, TEMA#4, SUBTEMA#5, SECTOR#6, AMBITO#7, TIPO_DE_INFORMACION#8, BASE_DE_REGISTRO#9, UNIDAD_DE_MEDIDA#10, PERIODO_INICIO#11, PERIODO_FINAL#12, FRECUENCIA#13, DIFUSION#14, MONTO#15]
Batched: false
Location: InMemoryFileIndex [file:/home/mauriciodiaz/MCD/1/PGBD/INtegrador/Integrador-PGBD-26A/data/raw/ingreso_gasto_finan.csv]
ReadSchema: struct<CICLO:int,MES:string,CLAVE_DE_CONCEPTO:string,NOMBRE:string,TEMA:string,SUBTEMA:string,SECTOR:string,AMBITO:string,TIPO_DE_INFORMACION:string,BASE_DE_REGISTRO:string,UNIDAD_DE_MEDIDA:string,PERIODO_INICIO:timestamp,PERIODO_FINAL:timestamp,FRECUENCIA:string,DIFUSION:string,MONTO:double>

(2) Project [codegen id : 1]
Output [17]: [CICLO#0, MES#1, CLAVE_DE_CONCEPTO#2, NOMBRE#3, TEMA#4, SUBTEMA#5, SECTOR#6, AMBITO#7, TIPO_DE_INFORMACION#8, BASE_DE_REGISTRO#9, UNIDAD_DE_MEDIDA#10, PERIODO_INICIO#11, PERIODO_FIN

In [9]:
filas_bronze = capa_bronze(spark)   # escribe data/bronze y valida que cada fuente tenga filas y columnas clave
m_bronze = metricas(spark, "bronze")   # se captura de inmediato: las celdas siguientes lanzan más jobs
display(pd.DataFrame([m_bronze]))
pd.DataFrame(
    [(n, filas_bronze[n], round(tamano_mb(r), 2), round(tamano_mb(BRONZE_DIR / n), 2)) for n, r in resolve_raw_paths().items()],
    columns=["fuente", "filas", "CSV (MB)", "Bronze Parquet (MB)"],
)

,etapa,jobs,stages,tareas,lectura_MB,escritura_MB,shuffle_KB,tiempo_tareas_s
0,bronze,20,20,42,228.79,4.83,0.9,60.37


,fuente,filas,CSV (MB),Bronze Parquet (MB)
0,shcp,746676,224.93,4.29
1,sat,201,0.04,0.04
2,igae,120,0.92,0.51
3,inpc,2002,0.40,0.02


> **Qué hizo Spark en Bronze:**
> - **Particiones:** el CSV de SHCP se dividió en **12 particiones de ~19 MB**, exactamente lo que predice la fórmula y una por núcleo. Los archivos pequeños (SAT, IGAE, INPC) quedan en 1 partición cada uno porque pesan menos que `openCostInBytes` (4 MB). Forzar *splits* más pequeños es más lento: más tareas que núcleos, cada una con su costo fijo de planificación (medido en la guía de optimización).
> - **Plan físico:** `Scan csv` con las **16 columnas** (Bronze conserva todo, así que no hay *column pruning*) y un `Project` que agrega `_archivo_origen` y `_fecha_ingesta`. **No hay `Exchange`**: Bronze es una transformación *narrow*; cada tarea lee su trozo y escribe su propio archivo Parquet, sin mover datos entre tareas. El asterisco (`*`) indica *whole-stage codegen*: lectura y proyección se compilan en una sola función.
> - **Spark UI:** ~20 jobs (escritura de las 4 fuentes, más el conteo y la revisión de columnas de `validate_raw`) y **una sola lectura del CSV (229 MB)**. El esquema explícito de SHCP evita la pasada adicional que haría `inferSchema`.
> - **Compresión:** los 225 MB de SHCP quedan en **~4.3 MB de Parquet (≈ 50 veces menos)**. Parquet guarda por columna y codifica con diccionario las columnas repetitivas (`NOMBRE`, `TEMA`, `UNIDAD_DE_MEDIDA`…), más compresión Snappy.
>
> **Por qué importa:** a partir de aquí nadie vuelve a leer el CSV. En la versión sin capas, las validaciones releían el CSV ~12 veces (2.7 GB por ejecución).

## 8. Capa Silver: limpieza, homologación y validación por fuente

Silver lee Bronze y deja una tabla limpia por fuente con la llave común `fecha` (primer día del mes): SHCP filtrado a claves XBC mensuales en pesos, deduplicado y convertido a millones; SAT sin marcas preliminares; IGAE de formato ancho a largo; INPC con su índice general. Después de escribir, las compuertas de calidad revisan **lo que quedó en disco**: sin nulos, sin duplicados por llave y 123 meses continuos en la ventana de estudio.

In [10]:
# Plan físico de la limpieza de SHCP leyendo desde Bronze
clean_all(read_layer(spark, BRONZE_DIR))["shcp"].explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [CICLO#6818, MES#6819, CLAVE_DE_CONCEPTO#6820, UNIDAD_DE_MEDIDA#6828, FRECUENCIA#6831, MONTO#6833]
Batched: true
Location: InMemoryFileIndex [file:/home/mauriciodiaz/MCD/1/PGBD/INtegrador/Integrador-PGBD-26A/data/bronze/shcp]
PushedFilters: [IsNotNull(CLAVE_DE_CONCEPTO), StringStartsWith(CLAVE_DE_CONCEPTO,XBC)]
ReadSchema: struct<CICLO:int,MES:string,CLAVE_DE_CONCEPTO:string,UNIDAD_DE_MEDIDA:string,FRECUENCIA:string,MONTO:double>

(2) Filter
Input [6]: [CICLO#6818, MES#6819, CLAVE_DE_CONCEPTO#6820, UNIDAD_DE_MEDIDA#6828, FRECUENCIA#6831, MONTO#6833]
Condition : ((((isnotnull(CLAVE_DE_CONCEPTO#6820) AND (trim(FRECUENCIA#6831, None) = Mensual)) AND StartsWith(CLAVE_DE_CONCEPTO#6820, XBC)) AND Contains(lower(trim(UNIDAD_DE_MEDIDA#6828, None)), pesos)) AND atleastnno

In [11]:
silver, reporte_silver = capa_silver(spark)
display(pd.DataFrame(reporte_silver).T)
m_silver = metricas(spark, "silver")
pd.DataFrame([m_silver])

,filas_ventana,meses_unicos,rango
shcp,3567,123,2015-01-01 a 2025-03-01
sat,123,123,2015-01-01 a 2025-03-01
igae,123,123,2015-01-01 a 2025-03-01
inpc,123,123,2015-01-01 a 2025-03-01


,etapa,jobs,stages,tareas,lectura_MB,escritura_MB,shuffle_KB,tiempo_tareas_s
0,silver,61,61,72,6.42,0.07,181.5,9.72


> **Qué hizo Spark en Silver:**
> - **`Scan parquet` lee solo 6 de las 18 columnas de Bronze** (`ReadSchema`). En Parquet esto es *column pruning* real: las columnas no usadas ni siquiera se leen del disco.
> - **`PushedFilters: StringStartsWith(CLAVE_DE_CONCEPTO, XBC)`**: el filtro de claves XBC se empuja al lector de Parquet, que puede descartar grupos de filas completos usando las estadísticas mínimo/máximo del archivo. `Batched: true` indica lectura vectorizada.
> - **`HashAggregate → Exchange hashpartitioning(..., 4) → HashAggregate`**: así implementa Spark `dropDuplicates()`, en **dos fases**. La fase parcial deduplica dentro de cada partición antes del shuffle; el `Exchange` reúne las filas iguales en la misma de **4 particiones** (`spark.sql.shuffle.partitions`), y la fase final termina el trabajo.
> - **`AdaptiveSparkPlan`**: AQE reoptimiza en cada frontera de stage con estadísticas reales.
> - **Spark UI:** lee **~6.4 MB** (contra 229 MB del CSV) y mueve **~180 KB** por shuffle. Los **~60 jobs** se deben a las compuertas de calidad: cada `assert` (nulos, rango de fechas, duplicados, meses distintos, conteo) es una acción sobre cada una de las cuatro fuentes, y se ejecuta sobre lo que ya quedó escrito en `data/silver/`.
>
> **Por qué 4 particiones de shuffle:** con ~180 KB por capa no hay volumen que repartir. El valor por defecto (200) crea cientos de tareas casi vacías y en las mediciones fue entre 40 % y 70 % más lento; entre 1 y 12 particiones el tiempo es equivalente (ver guía de optimización).

## 9. Capa Gold: dataset maestro analítico

Gold integra las cuatro tablas Silver en el dataset maestro: pivotea las 29 claves de SHCP (una columna por clave), une SAT, IGAE e INPC por `fecha`, calcula la recaudación real, los ratios fiscales y las variaciones anuales, y escribe Parquet particionado por año.

In [12]:
# Plan físico del dataset maestro leyendo desde Silver
build_master_dataset(silver).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (29)
+- Project (28)
   +- Window (27)
      +- Sort (26)
         +- Exchange (25)
            +- Project (24)
               +- BroadcastHashJoin Inner BuildRight (23)
                  :- Project (19)
                  :  +- BroadcastHashJoin Inner BuildRight (18)
                  :     :- Project (14)
                  :     :  +- BroadcastHashJoin Inner BuildRight (13)
                  :     :     :- Project (9)
                  :     :     :  +- HashAggregate (8)
                  :     :     :     +- Exchange (7)
                  :     :     :        +- HashAggregate (6)
                  :     :     :           +- HashAggregate (5)
                  :     :     :              +- Exchange (4)
                  :     :     :                 +- HashAggregate (3)
                  :     :     :                    +- Filter (2)
                  :     :     :                       +- Scan parquet  (1)
                  :     :     +- Broadca

In [13]:
maestro, reporte_gold, ruta_gold = capa_gold(spark, silver)
print(reporte_gold, "->", ruta_gold.relative_to(ROOT))
m_gold = metricas(spark, "gold")
display(pd.DataFrame([m_gold]))
maestro.select("fecha", "shcp_xbc", "sat_total_tributarios", "ratio_tributario_gasto",
               "igae_total", "inpc", "igae_var_anual_pct", "inflacion_anual_pct").show(5)

{'filas': 123, 'columnas': 43, 'rango': '2015-01-01 a 2025-03-01'} -> data/gold/dataset_maestro


,etapa,jobs,stages,tareas,lectura_MB,escritura_MB,shuffle_KB,tiempo_tareas_s
0,gold,40,40,50,0.53,0.17,163.8,1.79


+----------+-----------+---------------------+----------------------+-----------------+-----------+------------------+-------------------+
|     fecha|   shcp_xbc|sat_total_tributarios|ratio_tributario_gasto|       igae_total|       inpc|igae_var_anual_pct|inflacion_anual_pct|
+----------+-----------+---------------------+----------------------+-----------------+-----------+------------------+-------------------+
|2015-01-01|340847.0778|           241470.743|                0.7084|90.61837247449863|87.11010277|              NULL|               NULL|
|2015-02-01|271271.8257|           173802.732|                0.6407|89.26774023624809|87.27537713|              NULL|               NULL|
|2015-03-01|273661.9216|           250012.529|                0.9136|94.33154633109469|87.63071699|              NULL|               NULL|
|2015-04-01|274080.3724|           207142.087|                0.7558|92.92938410104547|87.40384038|              NULL|               NULL|
|2015-05-01|277618.8672|   

> **Qué hizo Spark en Gold:**
> - **Lectura mínima:** cada `Scan parquet` de Silver lee solo las columnas necesarias, y el filtro de la ventana 2015-01 a 2025-03 se empuja al lector (`PushedFilters: GreaterThanOrEqual(fecha, 2015-01-01), LessThanOrEqual(fecha, 2025-03-01)`).
> - **El `pivot` son dos agregaciones con su `Exchange`:** primero `hashpartitioning(fecha, CLAVE_DE_CONCEPTO)` toma el monto de cada par, y después `hashpartitioning(fecha)` arma **una fila por mes con 29 columnas**. Como `pivot()` no recibe la lista de claves, Spark lanza un job previo para descubrir los valores distintos (unos milisegundos sobre Silver).
> - **Tres `BroadcastExchange` + `BroadcastHashJoin ... BuildRight`:** SAT, IGAE e INPC (123 filas cada una) se convierten en tablas hash y se envían completas a cada tarea, así que el lado del pivot no se reparte por red. Con 123 filas el broadcast no ahorra tiempo medible, y Spark lo elegiría solo (cada tabla pesa < 40 KB, muy por debajo del umbral de 10 MB). El `F.broadcast()` deja explícita la intención para cuando el lado de SHCP crezca.
> - **`Exchange` (SinglePartition) → `Sort` → `Window`:** las variaciones anuales usan `lag(12)` sobre `Window.orderBy("fecha")`. Enero de 2016 necesita enero de 2015, así que la serie completa debe estar ordenada en **una sola partición**. Por eso aparece el aviso `No Partition Defined for Window operation`; con 123 filas el costo es despreciable, y particionar por año rompería el `lag` en cada cambio de año.
> - **Spark UI:** lee **~0.5 MB**. Los ~40 jobs corresponden a las validaciones del maestro, la escritura particionada por año y la relectura de verificación.

## 10. Resumen: qué hizo Spark en cada capa

In [14]:
resumen = pd.DataFrame([m_bronze, m_silver, m_gold]).set_index("etapa")
resumen.loc["total"] = resumen.sum()
resumen

,jobs,stages,tareas,lectura_MB,escritura_MB,shuffle_KB,tiempo_tareas_s
etapa,,,,,,,
bronze,20.0,20.0,42.0,228.79,4.83,0.9,60.37
silver,61.0,61.0,72.0,6.42,0.07,181.5,9.72
gold,40.0,40.0,50.0,0.53,0.17,163.8,1.79
total,121.0,121.0,164.0,235.74,5.07,346.2,71.88


> **Lectura del resumen:**
> - **Bronze concentra casi toda la lectura (~229 de ~236 MB) y la mayor parte del tiempo de tareas:** interpretar las 746,676 filas del CSV es el único trabajo pesado del proyecto. Después, el volumen baja a ~6 MB en Silver y a ~0.5 MB en Gold.
> - **El modelo medallion sustituye a la caché.** El pipeline original sin capas leía el CSV ~12 veces por ejecución (2.7 GB); con `persist()` bajó a 2 lecturas. Con capas materializadas en Parquet el CSV se lee **una sola vez**, y Silver y Gold pueden reprocesarse sin tocarlo.
> - **El shuffle total no llega a 0.5 MB**, lo que justifica pocas particiones de shuffle.
>
> Los experimentos comparativos (particiones de lectura, caché, 1/4/12/200 particiones de shuffle con y sin AQE, broadcast frente a sort-merge, memoria) y la tabla de decisiones están en [docs/guia_optimizacion.md](../docs/guia_optimizacion.md). Cada ejecución de `python -m src.pipeline` deja estas métricas por capa en `reports/evidencia_ejecucion_act11.txt`.

In [15]:
spark.stop()

### Estructura final del repositorio

```text
Integrador-PGBD-26A/
├── data/                           # En .gitignore: se regenera con este notebook o con src.pipeline
│   ├── raw/                        # Archivos descargados (CSV, XLS, ZIP)
│   ├── bronze/                     # Parquet sin transformar + metadatos de ingesta
│   ├── silver/                     # Una tabla limpia y validada por fuente
│   └── gold/dataset_maestro/       # Dataset maestro, Parquet particionado por año
├── docs/                           # Fuentes, arquitectura, roles y guía de optimización (resultados medidos)
├── notebooks/
│   ├── 01_ingesta.ipynb            # Descarga + Bronze → Silver → Gold, con el plan de Spark en cada capa
│   └── 02_analisis_dataset_maestro_final.ipynb  # Análisis sobre Gold
├── reports/                        # Evidencia de ejecución con métricas de Spark por capa
├── spark-events/                   # Event log para el History Server (en .gitignore)
├── src/
│   ├── pipeline.py                 # Orquestador medallion (python -m src.pipeline)
│   ├── monitoring.py               # Métricas de la Spark UI por etapa
│   ├── ingestion/loader.py         # Sesión Spark, esquemas, rutas de capas, Bronze
│   ├── quality/validators.py       # Compuertas de calidad
│   └── transformations/
│       ├── clean.py                # Silver: limpieza y homologación
│       └── integrate.py            # Gold: pivot, joins, ventanas y escritura
├── .gitignore
├── README.md
└── requirements.txt
```